# Projeto: 7 Dias de Código
## Importação de bibliotecas / Configuração de parâmetros

In [ ]:
from pathlib import Path
from typing  import Union, Optional
#
import logging as log
import numpy   as np
import os
import pandas  as pd
#
pd.options.display.max_rows = 10000
pd.options.display.max_columns = 30
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)

## Funções Genéricas do Projeto

In [ ]:

def fxPathFile(strPathFile: str, strFile: str, strExtFile: str) -> str:
    """
    Função..: fxPathFile
    Objetivo: Concatena o Caminho e o Nome do arquivo em uma única variável
            Caso o diretório não exista, ele será criado.
    """
    if not strPathFile:
        raise ValueError("O caminho base não pode ser vazio.")

    if not strFile:
        raise ValueError("O nome do arquivo não pode ser vazio.")

    # Cria o diretório caso não exista
    os.makedirs(strPathFile, exist_ok=True)

    # Garante extensão do arquivo
    if strExtFile and not strFile.lower().endswith(f".{strExtFile}"):
        strFile = f"{strFile}.{strExtFile}"

    return os.path.join(strPathFile, strFile)

def fxOpenFile(strFile: str):
    """
    Função..: fxOpenFile
    Objetivo: Abrir arquivos tipo parquet.
            Em caso de falha na abertura, retorna uma mensagem de erro específica para OSError
            e imprime a mensagem de erro para outros tipos de falha.
    """
    try:
        df = pd.read_parquet(strFile)
        return df
    except OSError as e:
        print(f"ERRO: Falha ao abrir o arquivo '{strFile}'. Verifique o caminho ou a integridade do arquivo. Detalhes: {e}")
        return None
    except Exception as e:
        print(f"ERRO: Ocorreu um erro inesperado ao abrir o arquivo '{strFile}'. Detalhes: {e}")
        return None

def fxSaveParquet(
    df: pd.DataFrame,
    caminho: Union[str, Path],
    verificar_integridade: bool = True,
    comprimir: Optional[str] = 'snappy'
) -> bool:
    """
    Função..: fxSaveParquet
    Objetivo: Salva um DataFrame em formato Parquet e verifica a gravação.
    Args....:
        df: DataFrame a ser salvo
        caminho: Caminho completo do arquivo (incluindo .parquet)
        verificar_integridade: Se True, valida a gravação lendo o arquivo
        comprimir: Tipo de compressão ('snappy', 'gzip', 'brotli' ou None)

    Returns:
        bool: True se gravação bem-sucedida, False caso contrário
    """
    # Validação inicial: DataFrame não pode estar vazio
    if df.empty:
        raise ValueError("DataFrame não pode estar vazio")

    # Converte para Path para manipulação consistente de caminhos
    caminho_arquivo = Path(caminho)

    # Garante que o arquivo tenha extensão .parquet
    if caminho_arquivo.suffix != '.parquet':
        caminho_arquivo = caminho_arquivo.with_suffix('.parquet')

    try:
        # Cria diretórios intermediários se não existirem
        caminho_arquivo.parent.mkdir(parents=True, exist_ok=True)

        # Salva o DataFrame em formato Parquet
        df.to_parquet(
            caminho_arquivo,
            compression=comprimir,
            index=False,
            engine='pyarrow'  # Engine padrão no Colab
        )

        # Verifica se o arquivo foi realmente criado no sistema
        if not caminho_arquivo.exists():
            log.error(f"Arquivo não foi criado: {caminho_arquivo}")
            return False

        # Validação de integridade: lê o arquivo e compara com original
        if verificar_integridade:
            df_validacao = pd.read_parquet(caminho_arquivo)

            # Verifica se dimensões (linhas x colunas) são iguais
            if df.shape != df_validacao.shape:
                log.error("Dimensões do arquivo salvo diferem do original")
                return False

            # Verifica se os nomes das colunas correspondem
            if not df.columns.equals(df_validacao.columns):
                log.error("Colunas do arquivo salvo diferem do original")
                return False

        # Exibe tamanho do arquivo salvo
        tamanho_mb = caminho_arquivo.stat().st_size / (1024 * 1024)
        log.info(f"✓ Arquivo salvo: {caminho_arquivo} ({tamanho_mb:.2f} MB)")

        return True

    except PermissionError:
        log.error(f"Sem permissão para escrever em: {caminho_arquivo}")
        return False

    except Exception as e:
        log.error(f"Erro ao salvar arquivo: {str(e)}")
        return False

def fxSaveDownload(
    df: pd.DataFrame,
    nome_arquivo: str = 'dados.parquet',
    baixar: bool = True
) -> bool:
    """
    Salva Parquet e oferece download automático no Colab.

    Args:
        df: DataFrame a ser salvo
        nome_arquivo: Nome do arquivo (sem necessidade de caminho)
        baixar: Se True, inicia download automaticamente

    Returns:
        bool: True se operação bem-sucedida
    """
    # Salva o arquivo no diretório temporário do Colab
    sucesso = fxSaveParquet(df, nome_arquivo)

    if sucesso and baixar:
        try:
            # Faz download do arquivo para máquina local
            files.download(nome_arquivo)
            log.info(f"Download iniciado: {nome_arquivo}")
        except Exception as e:
            log.error(f"Erro ao fazer download: {str(e)}")
            return False

    return sucesso


def fxRemoveFileIfExists(strPathFile: str) -> None:
    """
    Função..: fxRemoveFileIfExists
    Objetivo: Remover o arquivo caso ele exista.
    """
    if os.path.isfile(strPathFile):
        os.remove(strPathFile)

def fxAcertaDataHora(df, strNomeColuna):
    """
    Função - fxAcertaDataHora
    Objetivo.: Transformar as colunas: [data_emprestimo, data_devolucao, data_renovacao'] que possuem o
            formato: YYYY-mm-dd HH:MM:SS.ffffffff para o formato: YYYY-mm-dd HH:MM
    """
    intContReg  = 1
    lstDataHora = []
    #
    for conteudo in df[strNomeColuna]:
        if str(conteudo) == 'nan':
            conteudo = ''
        elif len(conteudo) > 16:
            conteudo = conteudo[:16]
        #
        lstDataHora.append(conteudo)
        print(f'Registro: {str(intContReg)}, Data Devolucao: {conteudo}')
        intContReg += 1
    #
    df[strNomeColuna] = lstDataHora
    return None

"""
# Função: Converte os valores de uma coluna STRING de um DataFrame para Data
"""
fxConvParaData = lambda df, strNomeColuna : pd.to_datetime(df[strNomeColuna])

"""
# Função: Converte o valor de uma Variável STRING para Data
"""
fxConvStrParaData = lambda strNomeColuna : pd.to_datetime(strNomeColuna, dayfirst=True)


## 1ª Etapa - Importação dos dados

## 1ª Etapa - Parte 1 - Definição de todas as Constantes

In [ ]:
class Config:

    # Informações dos arquivos
    SEP_CSV     = ','
    EXT_CSV     = 'csv'
    EXT_EXCEL   = 'xlsx'
    EXT_JSON    = 'json'
    EXT_PARQUET = 'parquet'

    PREFIXO_ARQ_CSV = 'emprestimos-'

    # Nomes dos bancos de dados
    DB_EMPR_BRONZE = 'DB_Empr_Bronze'
    DB_EMPR_SILVER = 'DB_Empr_Silver'
    DB_EMPR_SILVER_V2 = 'DB_Empr_Silver_V2'
    DB_EMPR_GOLD   = 'DB_Empr_Gold'
    DB_EMPR_DATA   = 'DB_Empr_Data'
    DB_EMPR_ANO    = 'DB_Empr_Ano'
    DB_EMPR_MES    = 'DB_Empr_Mes'
    DB_EMPR_HORA   = 'DB_Empr_Hora'
    DB_EMPR_DUPLICADOS    = 'DB_Empr_Duplicados'
    DB_EMPR_INCONSISTENTE = 'DB_Empr_Inconsistente'
    DB_EMPR_PERDIDOS       = 'DB_Empr_Perdidos'

    # Caminhos e URLs
    PATH_BASE: str = f'C:/Users/rtoni/OneDrive/Git-Dados/7DaysOfCode/'
    URL_CSV     = 'https://github.com/FranciscoFoz/7_Days_of_Code_Alura-Python-Pandas/blob/main/Dia_1-Importando_dados/Datasets/dados_emprestimos/'
    URL_PARQUET = 'https://github.com/FranciscoFoz/7_Days_of_Code_Alura-Python-Pandas/raw/main/Dia_1-Importando_dados/Datasets/dados_exemplares.parquet'
    URL_EXCEL   = 'https://github.com/FranciscoFoz/7_Days_of_Code_Alura-Python-Pandas/blob/raw/Dia_6-Novos_dados_novas_analises/Datasets/matricula_alunos.xlsx'

    # Parâmetros
    ANO_INICIAL  = 2010
    ANO_FINAL    = 2021
    ARQS_POR_ANO = 2

## 1ª Etapa - Parte 2 - Concatenando todos os arquivos CSV

In [ ]:
dicNomeDF = dict()
for intAno in range(2010, 2021):       # Faixa de Anos que serão analisados: 2010 - 2020
    for intQtdArqs in range(1, 3):     # Quantidade de arquivos por Ano = 2
        """
        # dicNomeDF  - Dicionário que armazena temporariamente os dados dos arquivos
        # strNomeDF  - Nome do DataFrame atual
        # strURL_Arq - Endereço da URL do GIT onde estão os dados
        """
        strNomeDF   = f'df_{str(intAno)}_{str(intQtdArqs)}'
        strURL_File = f'{Config.URL_CSV}{Config.PREFIXO_ARQ_CSV}{str(intAno)}{str(intQtdArqs)}.{Config.EXT_CSV}?raw=true'
        print(strURL_File)
        try:
            """
            # Processo de tratamento dos dados Livros Emprestados:
            # - Converte as colunas que contém Data/Hora para Datatime
            # - Converte as colunas ['id_emprestimo'] e [matricula_e_siape] para String
            """
            dicNomeDF[strNomeDF] = pd.read_csv(strURL_File, sep=Config.SEP_CSV)
        except Exception as e:
            print(strNomeDF, e)
            continue

print(dicNomeDF.values)

## 1ª Etapa - Parte 3 - Unificar todos os dicionários em um único arquivo. Grava o arquivo em disco

In [ ]:
bolResult     = False
lstDataFrames = []         # Lista que contém o(s) DataFrame(s) temporariamente
try:
    lstDataFrames.extend(dicNomeDF.values())
    strPathFile = fxPathFile(Config.PATH_BASE, Config.DB_EMPR_BRONZE, Config.EXT_PARQUET)
    df_Empr_Bronze = pd.concat(lstDataFrames, ignore_index = True)
    bolResult = fxSaveParquet(
        df = df_Empr_Bronze,
        caminho = strPathFile,
        verificar_integridade = True,
        comprimir = 'snappy'
    )
    if bolResult:
        print( f'Arquivo: {Config.DB_EMPR_BRONZE}.{Config.EXT_PARQUET} gravado com sucesso!!!' )

except OSError as e:
    print(e)

### Contagem total de registros / registros duplicados

In [ ]:

intQtdTotReg  = len(df_Empr_Bronze)
intQtdRegUni  = len(df_Empr_Bronze.value_counts())

print(f'Total de registros........: {intQtdTotReg:,} \n')
print(f'Total de registros únicos.: {intQtdRegUni:,} \n')

In [ ]:
df_Empr_Bronze.head()

In [ ]:
bolResult = False
df_Empr_Duplicados = df_Empr_Bronze[df_Empr_Bronze.duplicated(keep=False)].sort_values(by=df_Empr_Bronze.columns.tolist()).reset_index(drop=True)

try:
    strPathFile = fxPathFile(Config.PATH_BASE, Config.DB_EMPR_DUPLICADOS, Config.EXT_PARQUET)
    bolResult = fxSaveParquet(
        df = df_Empr_Duplicados,
        caminho = strPathFile,
        verificar_integridade = True,
        comprimir = 'snappy'
    )
    if bolResult:
        print( f'Arquivo: {Config.DB_EMPR_DUPLICADOS}.{Config.EXT_PARQUET} gravado com sucesso!!!' )
except OSError as e:
    print(e)

In [ ]:
print(f'Total de registros duplicados: {df_Empr_Bronze.duplicated().sum():,} no arquivo: {Config.DB_EMPR_BRONZE}.{Config.EXT_PARQUET} \n')
print(f'Total de registros duplicados: {len(df_Empr_Duplicados):,} no arquivo: {Config.DB_EMPR_DUPLICADOS}.{Config.EXT_PARQUET}')

### Registros que não possuem Número de matrícula ou SIAPE

In [ ]:
# Importante: Não foi encontrado nenhum registro que a coluna [data_emprestimo] ou [data_devolucao] possua valor nulo (NaN)

bolResult = False
try:
    strPathFile = fxPathFile(Config.PATH_BASE, Config.DB_EMPR_INCONSISTENTE, Config.EXT_PARQUET)
    df_Empr_Inconsistente = df_Empr_Bronze[df_Empr_Bronze['matricula_ou_siape'].isna()]
    #
    if len(df_Empr_Inconsistente) > 0:
        bolResult = fxSaveParquet(
        df = df_Empr_Inconsistente,
        caminho = strPathFile,
        verificar_integridade = True,
        comprimir = 'snappy'
    )

    if bolResult:
        print( f'Arquivo: {Config.DB_EMPR_INCONSISTENTE}.{Config.EXT_PARQUET} gravado com sucesso!!!' )
    print(f'Qtd Total de registros...........: {len(df_Empr_Inconsistente)}')

except OSError as e:
    print(e)

# Comando para selecionar os registros inconsistentes
# Quando uma ou mais colunas possuírem valores nulos (NaN) deve-se utilizar o operador OR (|)

# df_Inconsistente = df_Empr_Bronze[
    # df_Empr_Bronze['matricula_ou_siape'].isna()] |
    # df_Empr_Bronze['data_emprestimo'].isna()     |
    # df_Empr_Bronze['codigo_barras'].isna()
# ]

In [ ]:
df_Empr_Inconsistente.head()

In [ ]:
df_Empr_Bronze.count()

## 1ª Etapa - Parte 4 - Tratando Dataframe: df_Empr_Bronze
- Realizar a contagem dos registros
- Eliminar os registros duplicados, mantendo a primeira ocorrência
- Resultado gerar o df_Empr_Silver

In [ ]:
# Eliminar registros duplicados
bolResult = False
try:
    strPathFile = fxPathFile(Config.PATH_BASE, Config.DB_EMPR_SILVER, Config.EXT_PARQUET)
    #
    # Verifica a quantidade total de registros duplicados e os apaga
    df_Empr_Silver = df_Empr_Bronze.drop_duplicates(
        # subset = [
            # 'matricula_ou_siape',
            # 'data_emprestimo',
            # 'data_devolucao'
        # ],
        keep = 'first'
    )

    bolResult = fxSaveParquet(
        df = df_Empr_Silver,
        caminho = strPathFile,
        verificar_integridade = True,
        comprimir = 'snappy'
    )
    if bolResult:
        print( f'Arquivo: {Config.DB_EMPR_SILVER}.{Config.EXT_PARQUET} gravado com sucesso!!! \n' )
        print(f'Total de registros após tratamento de dados.......: {len(df_Empr_Silver):,} \n')
        print(f'Total de registros únicos após tratamento de dados: {len(df_Empr_Silver.value_counts()):,}')

except OSError as e:
    print(e)


try:
    strPathFile = fxPathFile(Config.PATH_BASE, Config.DB_EMPR_INCONSISTENTE, Config.EXT_PARQUET)
    df_Empr_Inconsistente = df_Empr_Bronze[df_Empr_Bronze['matricula_ou_siape'].isna()]
    #
    if len(df_Empr_Inconsistente) > 0:
        bolResult = fxSaveParquet(
        df = df_Empr_Inconsistente,
        caminho = strPathFile,
        verificar_integridade = True,
        comprimir = 'snappy'
    )

    if bolResult:
        print( f'Arquivo: {Config.DB_EMPR_INCONSISTENTE}.{Config.EXT_PARQUET} gravado com sucesso!!!' )
    print(f'Qtd Total de registros...........: {len(df_Empr_Inconsistente)}')

except OSError as e:
    print(e)


## 1ª Etapa - Parte 5 - Importando o arquivo: dados_exemplares.parquet
- Importar o arquivo e gravá-lo como CSV
- Incluir seu conteúdo no arquivo: DB_Emprestimo.csv